In [2]:
import pandas as pd

# Load the Excel file and inspect sheet names
file_name = 'Answer only prod.xlsx'
# file_name = 'Paragraph CoT prod.xlsx'
# file_name = 'Step-by-step CoT -- All at once prod.xlsx'
# file_name = 'Step-by-step CoT -- Sequential prod.xlsx'
xlsx = pd.ExcelFile(file_name)
print("Available sheets:", xlsx.sheet_names)

# Select the 'main study' sheet (or the first matching name)
sheet = next((s for s in xlsx.sheet_names if 'main study' in s.lower()), xlsx.sheet_names[0])
print("Using sheet:", sheet)

# Read the data
df = pd.read_excel(xlsx, sheet_name=sheet)

# Compute correctness according to the rules:
# 1) If Model Answer == Gt Answer, then 'Accept' is correct.
# 2) If Model Answer != Gt Answer, then 'Reject' is correct.
df['correct'] = ((df['Question idx'] != 1) &  
                 (df['Question idx'] != 2)  & 
                 (df['Question idx'] != 3) &  
                 (df['Question idx'] != 4) &
    (((df['Model Answer'] == df['Gt Answer']) & (df['Step 2'] == 'Accept')) | ((df['Model Answer'] != df['Gt Answer']) & (df['Step 2'] == 'Reject')))
)

df['overreliance'] = ((df['Question idx'] != 1) & (df['Question idx'] != 2)  & (df['Question idx'] != 3) & (df['Question idx'] != 4) &
                    (df['Model Answer'] != df['Gt Answer']) & (df['Step 2'] == 'Accept'))

# Summary statistics
total = len(df) - df['Question idx'].isin([1, 2, 3, 4]).sum()
correct_count = df['correct'].sum()
accuracy = correct_count / total * 100
overreliance_count = df['overreliance'].sum()

print(file_name)
print(f"Total test questions: {total}")
print(f"Correct: {correct_count}")
print(f"Overreliance: {overreliance_count}")
print(f"Overreliance rate: {overreliance_count / total * 100:.2f}%")
print(f"Accuracy: {accuracy:.2f}%")

# Show the first few rows with the new 'correct' column
# from ace_tools import display_dataframe_to_user
# display_dataframe_to_user("Preview with Correctness", df.head())


Available sheets: ['Task Demand Questions', 'AI Usage Questions', 'Free Form Questions', 'Interaction Questions', 'Demographics', 'Main Study', 'Evaluation', '679334a846e22adcedd4cfd0', '65fb580ab7fbb2d518ee8eaa', '6742d74e9f7b3b029a3791a6', '67e8bedcc2e4bb440dad3d65', '67f1519c5d31be4b27c93b8f', '677eed89918027fb1dc8b44f', '5c50046b2b89f500015e5f6b']
Using sheet: Main Study
Answer only prod.xlsx
Total test questions: 48
Correct: 43
Overreliance: 2
Overreliance rate: 4.17%
Accuracy: 89.58%


In [ ]:
import pandas as pd

def calculate_accuracy_by_user(
    filename: str,
    sheet_name: str = None
) -> pd.DataFrame:
    # 1. Load workbook & pick the “Main Study” sheet
    xlsx = pd.ExcelFile(filename)
    if sheet_name is None:
        # pick the first sheet whose name contains “main study” (case‐insensitive)
        sheet_name = next(
            s for s in xlsx.sheet_names
            if "main study" in s.lower()
        )
    df = pd.read_excel(xlsx, sheet_name=sheet_name)
    df = (
        df.groupby("Username", group_keys=False)
          .apply(lambda g: g.iloc[4:])   # remove first 4 rows in each group
    )

    # 2. Apply correctness rules
    df['correct'] = ((df['Question idx'] != 1) &  
                 (df['Question idx'] != 2)  & 
                 (df['Question idx'] != 3) &  
                 (df['Question idx'] != 4) &
    (((df['Model Answer'] == df['Gt Answer']) & (df['Step 2'] == 'Accept')) | ((df['Model Answer'] != df['Gt Answer']) & (df['Step 2'] == 'Reject'))))
    
    df['overreliance'] = ((df['Question idx'] != 1) & (df['Question idx'] != 2)  & (df['Question idx'] != 3) & (df['Question idx'] != 4) &
                    (df['Model Answer'] != df['Gt Answer']) & (df['Step 2'] == 'Accept'))

    # 3. Group by Username and compute aggregates
    summary = (
        df.groupby('Username')
          .agg(
              total=('correct', 'size'),
              correct=('correct', 'sum'),
              overreliance = ('overreliance', 'sum')
          )
          .reset_index()
    )
    summary['accuracy (%)'] = summary['correct'] / (summary['total']) * 100
    summary['overreliance (%)'] = summary['overreliance'] / (summary['total']) * 100

    return summary

if __name__ == "__main__":
    # point to your file here
    FPATH = "Answer only prod.xlsx"
    # FPATH = 'Paragraph CoT prod.xlsx'
    # FPATH = 'Step-by-step CoT -- All at once prod.xlsx'
    # FPATH = 'Step-by-step CoT -- Sequential prod.xlsx'

    result = calculate_accuracy_by_user(FPATH)
    print(FPATH)
    print(result)

    # optional: save to Excel
    result.to_excel(f'{FPATH}_accuracy_by_user.xlsx', index=False)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2708233868.py, line 22)